In [0]:
# ========================================
# 01_setup_environment
# ========================================
 
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime
 
# ---------------------------------------
# Base Paths
# ---------------------------------------
 
base_path = "/Volumes/healthcare_enterprise_project/heathcare_schema/healthcare_volume"
 
source_path = f"{base_path}/source_file"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"
quarantine_path = f"{base_path}/quarantine"
checkpoint_path = f"{base_path}/checkpoints"
logs_path = f"{base_path}/logs"
archive_path = f"{base_path}/archive"
 
# ---------------------------------------
# Create Audit Schema
# ---------------------------------------
 
audit_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("load_time", TimestampType(), True)
])
 
# ---------------------------------------
# Create Empty Audit Table
# ---------------------------------------
 
empty_df = spark.createDataFrame([], audit_schema)
 
empty_df.write.format("delta") \
    .mode("overwrite") \
    .save(f"{logs_path}/audit_logs")
 
# ---------------------------------------
# Audit Logging Function
# ---------------------------------------
 
def log_audit(
    pipeline_name,
    layer,
    table_name,
    record_count,
    status,
    error_message=""
):
 
    audit_data = [(
        pipeline_name,
        layer,
        table_name,
        record_count,
        status,
        error_message,
        datetime.now()
    )]
 
    columns = [
        "pipeline_name",
        "layer",
        "table_name",
        "record_count",
        "status",
        "error_message",
        "load_time"
    ]
 
    audit_df = spark.createDataFrame(audit_data, columns)
 
    audit_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{logs_path}/audit_logs")
 
# ---------------------------------------
# Success Message
# ---------------------------------------
 
print("Setup Completed Successfully")